# Circular Task Analysis — Assignment Notebook
## Report

This report presents the main issues we encountered during the development and analysis process, along with the underlying causes and the solutions we implemented. For each problem, we detail what went wrong, why it happened, and how we resolved it in order to ensure a correct and reliable workflow.

<details>
<summary><strong> Summary of Issues Encountered</strong></summary>

### 1. [Time Conversion](#1-time-conversion)
- [Problem](#problem-time)
- [Cause](#cause-time)
- [Resolution](#resolution-time)

### 2. [Formulas in the Markers Table](#2-formulas-in-the-markers-table)
- [Problem](#problem-formulas)
- [Cause](#cause-formulas)
- [Resolution](#resolution-formulas)

### 3. [Retrieve the Values from the Markers Table](#3-retrieve-the-values-from-the-markers-table)
- [Problem](#problem-values)
- [Cause](#cause-values)
- [Resolution](#resolution-values)

### 4. [Mask with Our Data](#4-mask-with-our-data)
- [Problem](#problem-mask)
- [Cause](#cause-mask)
- [Resolution](#resolution-mask)

### 5. [The initial organization of the Git repository](#5-the-initial-organization-of-the-git-repository)
- [Problem](#problem-mask)
- [Cause](#cause-mask)
- [Resolution](#resolution-mask)

### 6. [Incorrect File Paths in the Main Notebook](#6-incorrect-file-paths-in-the-main-notebook)
- [Problem](#problem-mask)
- [Cause](#cause-mask)
- [Resolution](#resolution-mask)

</details>


## 1. Time Conversion
### <a id="problem-time"></a>Problem
The time values in the dataset did not correspond to a normal time scale (seconds).

When plotting them directly, the x-axis appeared broken, extremely large, and unusable.

### <a id="cause-time"></a>Cause
The timestamps recorded in the CSV file were absolute system timestamps, corresponding to the computer’s internal clock (i.e., milliseconds since the Unix epoch in 1970).

This is not “time since the start of the experiment” but **“time since 1 January 1970”** — so around 1.6 trillion milliseconds, which completely breaks the plot.


### <a id="resolution-time"></a>Resolution
We converted the absolute timestamps into relative experiment time by:
1.	Taking the first timestamp $t0$
2.	Subtracting it from the whole time series ($to$ set time = $0$ at the beginning)
3.	Dividing by $1000$ to convert milliseconds → seconds
    
$$
t = \frac{\text{timestamps\_ms} - t_0}{1000}
$$


This produced a clean, correct time axis from 0 to ~180 seconds.

## 2. Formulas in the Markers Table
### <a id="problem-formulas"></a>Problem
The marker table only contained variable names (e.g., $nLaps$, $Re$, $Te$, $IDe$) without any formulas.

We needed to reconstruct all metrics from scratch.

### <a id="cause-formulas"></a>Cause
No formulas were provided in the HTML table.

We only had the names of the variables and their numeric output, but not the computation rules behind them.

Some metrics were easy to guess (mean radius, standard deviation, etc.),
but others especially $Te$ were non-standard and much harder to infer.
### <a id="resolution-formulas"></a>Resolution
Two strategies were used:
- Using the poster + AI assistance, we reconstructed most formulas
- For the most complex variable ($Te$), we validated the formula with Tifenn Fauviaux, who helped confirm that the correct expression was:


$Te = \sigma \cdot \sqrt{2 \pi e}$


where *e* is the **Euler number** (≈ 2.71828).

With this, all computed values finally aligned with the marker table.

## 3. Retrieve the Values from the Markers Table
### <a id="problem-values"></a>Problem
Our computed values ($Re$, $Te$, $MT$, $IPe$, etc.) did not match the values shown in the HTML marker table, even though the formulas seemed correct.
### <a id="cause-values"></a>Cause
We initially computed all metrics using all data points inside each record, including the beginning of the recording where the cursor was not yet inside the target.

However, the original software (MouseReMoCo) applies an internal rule:

 Statistics are computed only AFTER the first entry into the target.

We were mistakenly including the “approach” phase, which greatly distorted:
- $Re$ (radius too variable)
- $Te$ (variance too high)
- $nLaps$ (wrong angular integration)
- $MT$/$lap$
- $IPe$
- $Error%$
- $Be$

### <a id="resolution-values"></a>Resolution
With Lucas (another classmate) we realized the time series had to be filtered like this: 

inside = (in_rec == 1)

first_inside_idx = np.where(inside)[0][0]

t = t[first_inside_idx:]

x = x[first_inside_idx:]

y = y[first_inside_idx:]

in_rec = in_rec[first_inside_idx:]

Once we applied this filtering, all computed values finally matched the marker table.

## 4. Mask with Our Data
### <a id="problem-mask"></a>Problem
Between two records (e.g., Record 1 end → Record 2 start), the trajectory contained noise samples, which created overlapping plots and incorrect values.
### <a id="cause-mask"></a>Cause
The raw CSV contains continuous recording data, not 5 isolated segments.
So between each “Record” and “Pause”, there are background movements and noise.

If these were not removed, they interfered with:
- plot aesthetics
- $nLaps$
- $MT$/$lap$
- error estimates
- inside/outside segmentation

### <a id="resolution-mask"></a>Resolution
We masked each record separately using the timestamps extracted from the marker file:

mask = (timestamps >= start) & (timestamps <= end)

This ensured that each record only contained the exact 20-second task window, with no undesired samples.

After masking, the trajectory plots became clean, and metric computations became fully consistent with the recorded values.

## 5. The initial organization of the Git repository
### <a id="problem-mask"></a>Problem
The branch names and notebook titles were unclear, and the overall structure differed from one branch to another. This inconsistency made the repository difficult to navigate and prevented us from maintaining a clean and well-organized project.

### <a id="cause-mask"></a>Cause
The problem was caused by unclear naming choices at the start and by a lack of communication within the group.

### <a id="resolution-mask"></a>Resolution
We renamed the branches using simpler names (Figures, Markers, Report) and agreed on a working convention: creating a 001 notebook to work with the provided CSV files, and a 002 notebook for our own data exports.

## 6. Incorrect File Paths in the Main Notebook
### <a id="problem-mask"></a>Problem
When creating the main notebook meant to execute all other notebooks at once, the file paths used to load the CSV files were incorrect, which prevented the code from running properly.

### <a id="cause-mask"></a>Cause
The path to access the CSV files was different depending on whether the code was executed from inside the notebooks folder (requiring a ../ to go back one level) or from the repository’s root directory. This difference caused inconsistencies in how the files were loaded.

### <a id="resolution-mask"></a>Resolution
We removed the “step back” (../) from the file paths in the final main notebook. As a result, the main notebook now runs correctly from the root directory, while each individual notebook remains executable within its own branch.